# Day 8 — 분포와 요약통계: 숫자 하나로 줄이기 전에 모양부터 보기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-08.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-08.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 대칭 분포와 치우친 분포의 요약통계

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)          # seed 고정: 누가 돌려도 같은 난수
normal = rng.normal(loc=100, scale=15, size=200)        # 종 모양 분포
skewed = rng.lognormal(mean=4.5, sigma=0.6, size=200)   # 오른쪽 꼬리가 긴 분포

df = pd.DataFrame({"normal": normal, "skewed": skewed})

summary = pd.DataFrame({
    "mean": df.mean(),
    "median": df.median(),
    "std": df.std(),                                  # 표본 표준편차 (ddof=1)
    "IQR": df.quantile(0.75) - df.quantile(0.25),
})
print(summary.round(1))

### 예시 2 — 이상치 한 개의 위력

In [ ]:
import numpy as np

clean = np.array([10, 11, 12, 12, 13, 14])
dirty = np.append(clean, 60)             # 이상치 하나 추가

def iqr(x):
    q1, q3 = np.percentile(x, [25, 75])
    return q3 - q1

for name, x in [("이상치 없음", clean), ("이상치 있음", dirty)]:
    print(f"{name}: 평균={x.mean():.1f}, 중앙값={np.median(x):.1f}, "
          f"표준편차={x.std(ddof=1):.1f}, IQR={iqr(x):.1f}")

### 예시 3 — IQR 규칙으로 이상치 후보 찾기

In [ ]:
import numpy as np

x = np.array([10, 11, 12, 12, 13, 14, 60])
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"울타리: {lower} ~ {upper}")
print("이상치 후보:", x[(x < lower) | (x > upper)])

## 직접 해보기

1. JupyterLab(또는 VS Code)을 열고 `notebooks/` 폴더에 `02_statistics_basics.ipynb`를 새로 만듭니다. 첫 셀은 Markdown으로 `# Day 8 — 분포와 요약통계`라고 제목을 답니다.
2. 예시 1의 코드를 한 셀에 붙여 넣고 실행해 같은 표가 나오는지 확인합니다.
3. 다음 셀에서 두 분포를 histogram으로 그리고, 평균(빨간 실선)과 중앙값(검은 점선)을 표시합니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["normal", "skewed"]):
    sns.histplot(df[col], bins=30, ax=ax)
    ax.axvline(df[col].mean(), color="red", label="mean")
    ax.axvline(df[col].median(), color="black", linestyle="--", label="median")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.savefig("../figures/day08_distributions.png", dpi=150)
plt.show()

어느 그림에서 두 선이 벌어지는지, 벌어지는 방향이 꼬리 방향과 같은지 Markdown 셀에 한 문장으로 적습니다.
4. 예시 2와 예시 3을 각각 새 셀에서 실행합니다. 그다음 `dirty`의 60을 20으로 바꿔 다시 실행하고, 평균·표준편차가 얼마나 달라지는지, 20은 여전히 이상치 후보인지 확인합니다.
5. 이제 "그룹당 6마리"인 작은 실험을 흉내 냅니다. 예시 1의 `rng`를 이어서 씁니다.

In [ ]:
small = pd.DataFrame({
    "group": ["GC"] * 6 + ["FLT"] * 6,
    "expression": np.r_[rng.normal(10, 1, 6), rng.normal(11, 1, 6)],
})
small.loc[11, "expression"] = 17      # 일부러 넣은 이상치

print(small.groupby("group")["expression"].agg(["mean", "median", "std"]).round(2))

fig, ax = plt.subplots(figsize=(4, 4))
sns.boxplot(data=small, x="group", y="expression", color="lightgray", ax=ax)
sns.stripplot(data=small, x="group", y="expression", color="black", size=7, ax=ax)
plt.tight_layout()
plt.savefig("../figures/day08_small_sample.png", dpi=150)
plt.show()